### 参考链接

[构建一个基本聊天机器人](https://langgraph.com.cn/tutorials/get-started/1-build-basic-chatbot/index.html)

# 1、创建一个 StateGraph

In [ ]:
from typing import Annotated

from typing_extensions import TypedDict

# START的定义是：START = sys.intern("__start__")
# sys.intern() 的作用是：把一个字符串放进全局字符串池，之后所有相同内容的字符串都复用同一个对象，而不是新建
# 它本质就是一个被 intern 优化过的普通字符串
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages

In [ ]:
class State(TypedDict):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages: Annotated[list, add_messages]


graph_builder = StateGraph(State)

#### 图可以处理两个关键任务：
一、每个节点可以接收当前状态作为输入，并输出状态的更新

二、对消息的更新将追加到现有列表而不是覆盖

# 2、添加一个节点

#### 接下来，添加一个“chatbot”节点。 节点 表示工作单元，通常是普通的 Python 函数。

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
# from langchain.chat_models import init_chat_model


# 加载 .env 文件
load_dotenv(override=True)

# 从环境变量读取，显式传给 ChatOpenAI
llm = ChatOpenAI(
    model="mimo-v2-pro",
    api_key=os.getenv("XIAOMI_API_KEY"),
    base_url=os.getenv("XIAOMI_BASE_URL"),
)

#### 将聊天模型集成到一个简单的节点中

In [ ]:
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

"""
逐一拆解这行代码：

```python
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}
```

---

### 1. 为什么 `return {"messages": ...}`

节点函数的返回值会被 LangGraph **合并进全局 State**。State 里定义了 `messages` 字段，所以这里必须用相同的 key 名 `"messages"` 来告诉 LangGraph："我要往 State 的 messages 字段里写数据。"

如果 State 定义了其他字段，你也可以同时更新它们：

```python
class State(TypedDict):
    messages: Annotated[list, add_messages]
    turn_count: int

def chatbot(state: State):
    return {
        "messages": [llm.invoke(state["messages"])],
        "turn_count": 1,            # 同时更新 turn_count
    }
```

返回的字典 key 必须是 State 里已声明的字段名，否则 LangGraph 不认识。

---

### 2. 为什么 value 是一个列表

因为 `messages` 字段在 State 里定义的是 `list` 类型，而 Reducer `add_messages` 的合并逻辑是"把新列表追加到旧列表末尾"。所以你返回的必须是一个列表，即使里面只有一个元素：

```python
# 正确：返回列表，里面有一个 AIMessage
{"messages": [AIMessage(content="你好")]}

# 错误：直接返回消息对象，不是列表，类型不匹配
{"messages": AIMessage(content="你好")}
```

可以这样理解：`messages` 是一个"消息收件箱"，你往里面扔东西，必须用列表包起来再扔。

---

### 3. 为什么是 `llm.invoke(...)`

`llm.invoke()` 是 LangChain 的 Runnable 协议标准调用方式。等价于"让模型处理这段输入并返回结果"：

```python
# invoke 的输入：一个消息列表
# invoke 的输出：一个 AIMessage 对象
result = llm.invoke(state["messages"])
# result 是 AIMessage(content="你好，有什么可以帮你？", ...)
```

`invoke` 内部实际做了这些事：
1. 把消息列表转换成 API 请求格式
2. 调用大模型的 HTTP API
3. 把返回结果解析成 `AIMessage` 对象

---

### 4. 为什么参数是 `state["messages"]`

大模型需要看到完整的对话历史才能做出合理回应。`state["messages"]` 里存着从对话开始到现在的所有消息：

```python
# 假设当前 state["messages"] 是这样的：
[
    HumanMessage("你好"),
    AIMessage("你好，有什么可以帮你？"),
    HumanMessage("今天天气怎么样？"),    # 用户最新的问题
]

# llm.invoke(state["messages"]) 会把这整段历史发给模型
# 模型看到完整上下文后，才能生成有连贯性的回复
```

如果只传最新一条消息（`state["messages"][-1]`），模型就失去了历史记忆，变成"失忆状态"。

---

### 完整数据流一图看懂

```
用户输入 "今天天气怎么样"
        ↓
State: {messages: [HumanMessage("今天天气怎么样")]}
        ↓
chatbot 节点被调用，收到 state
        ↓
llm.invoke(state["messages"])  →  发给大模型  →  AIMessage("今天是晴天，适合出门")
        ↓
返回 {"messages": [AIMessage("今天是晴天，适合出门")]}
        ↓
Reducer add_messages 将其追加到 State
        ↓
State: {messages: [HumanMessage("今天天气怎么样"), AIMessage("今天是晴天，适合出门")]}
        ↓
流转到下一个节点（或 END）
```

---

### 总结

| 部分                       | 作用                                          |
| -------------------------- | --------------------------------------------- |
| `return {"messages": ...}` | 声明要更新 State 的哪个字段                   |
| `[...]` 包成列表           | 匹配 State 的 list 类型，Reducer 期望接收列表 |
| `llm.invoke(...)`          | 调用大模型，发送输入，拿到回复                |
| `state["messages"]`        | 传入完整对话历史，让模型有上下文记忆          |
"""

# The first argument is the unique node name
# The second argument is the function or object that will be called whenever
# the node is used.
graph_builder.add_node("chatbot", chatbot)

# 3、添加一个 入口 节点

添加一个 入口 节点，以告诉图每次运行时**从何处开始工作**

In [ ]:
graph_builder.add_edge(START, "chatbot")

# 4、编译图

运行图之前，我们需要对其进行编译。我们可以通过在图构建器上调用 compile() 来完成。这将创建一个 CompiledGraph，我们可以在我们的状态上调用它

In [ ]:
graph = graph_builder.compile()

# 5、可视化图（可选）

您可以使用 get_graph 方法和其中一个“绘图”方法（例如 draw_ascii 或 draw_png）来可视化图。这些 draw 方法都需要额外的依赖项

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    # This requires some extra dependencies and is optional
    pass

# 6、运行聊天机器人

In [ ]:
graph.invoke({"messages": [{"role": "user", "content": "请介绍一下你自己"}]})

In [10]:
def stream_graph_updates(user_input: str):
    # stream：流式调用，图每执行完一个节点，就立刻返回一个中间结果

    # 只要调用 graph.stream() 或 graph.invoke()，就一定是从 START 节点开始执行，不需要你手动指定。
    # 原因很简单——add_edge(START, "chatbot") 这行代码已经把 START 和 chatbot 连起来了:
    # graph_builder.add_edge(START, "chatbot")
    # LangGraph 在编译图时会检查：哪个节点有从 START 出发的边，那个节点就是入口节点。
    # 你之前传入的 dict 会先经过 START 节点，START 节点把数据原样传给下一个节点（chatbot），然后
    # chatbot 才真正开始处理。
    for event in graph.stream({"messages": [{"role": "user", "content": user_input}]}):
        # value 就是当前节点执行完之后的完整 State
        for value in event.values():
            print("Assistant:", value["messages"][-1].content)

while True:
    try:
        user_input = input("User: ")
        if user_input.lower() in ["quit", "exit", "q"]:
            print("Goodbye!")
            break
        stream_graph_updates(user_input)
    except:
        # fallback if input() is not available
        user_input = "What do you know about LangGraph?"
        print("User: " + user_input)
        stream_graph_updates(user_input)
        break

"""
## 问题 1：`stream` vs `invoke` 的区别

```python
# invoke：同步调用，等整个图跑完，一次性返回最终结果
result = graph.invoke({"messages": [{"role": "user", "content": "你好"}]})
# 整个图执行完毕后，才拿到 result

# stream：流式调用，图每执行完一个节点，就立刻返回一个中间结果
for event in graph.stream({"messages": [{"role": "user", "content": "你好"}]}):
    # 每个节点执行完，都会触发一次这里的代码
    print(event)
```

关键区别：

|          | `invoke`                 | `stream`                      |
| -------- | ------------------------ | ----------------------------- |
| 返回时机 | 全部跑完才返回           | 每个节点跑完就返回一次        |
| 返回值   | 最终 State（一个 dict）  | 一个生成器，每次产出一个 dict |
| 用户体验 | 要等，然后一次性看到结果 | 边跑边看，体感更快            |
| 适用场景 | 后台批处理               | 对话机器人（需要实时反馈）    |

这里用 `stream` 是为了**实时打印模型的回复**，而不是等整个图跑完才显示。如果换成 `invoke` 完全可以，只是用户要等更久才能看到输出。

---

## 问题 2：`event` 代表什么

`graph.stream()` 每产出一个 `event`，代表**图中某个节点刚刚执行完毕**。这个 `event` 是一个 dict，key 是节点名，value 是该节点执行后更新的 State：

```python
# 假设图有三个节点：chatbot -> tool -> chatbot
# stream 会依次产出：

# event 1: chatbot 节点执行完
{"chatbot": {"messages": [HumanMessage("你好"), AIMessage("我来帮你查一下...")]}}
#  key是节点名   value是该节点执行后的完整State

# event 2: tool 节点执行完
{"tool": {"messages": [HumanMessage("你好"), AIMessage("..."), ToolMessage("查询结果")]}}

# event 3: chatbot 再次执行完
{"chatbot": {"messages": [..., AIMessage("根据查询结果，答案是...")]}}
```

所以 `event` 就是**"某个节点执行完毕的通知"**，里面带着该节点产出的最新 State。

---

## 问题 3：为什么还要 `event.values()` 再遍历一次

因为 `event` 的结构是 `{节点名: State}`，用 `.values()` 取出 State：

```python
event = {"chatbot": {"messages": [..., AIMessage("你好啊")]}}
#        ^key        ^value (这就是State)

# .values() 取出所有 value（这里只有一个，因为每次只有一个节点执行完）
for value in event.values():
    # value 就是 {"messages": [..., AIMessage("你好啊")]}
    # value["messages"] 是完整的消息列表
    # value["messages"][-1] 取最后一条，即模型刚生成的回复
    print("Assistant:", value["messages"][-1].content)
```

为什么不直接 `print(event["chatbot"]["messages"][-1].content)`？

因为图里可能有多个节点，节点名不固定。用 `.values()` 可以**不关心节点叫什么名字，直接拿 State**：

```python
# 如果图里只有一个节点（比如这个简单例子），这样写更直观
# 但通用写法是 .values()，这样无论图有多少节点都能用
```

---

## 完整执行流程一图看懂

```
用户输入: "你好"
        ↓
graph.stream({"messages": [{"role": "user", "content": "你好"}]})
        ↓
event 1 产出: {"chatbot": {messages: [HumanMessage("你好"), AIMessage("嗨！")]}}
            → values() 取出 → {"messages": [...]}
            → [-1] 取最后一条 → AIMessage("嗨！")
            → 打印: "Assistant: 嗨！"
        ↓
event 2 产出: {"tool_node": {messages: [..., ToolMessage("天气查询结果")]}}
            → values() 取出 → {"messages": [...]}
            → [-1] 取最后一条 → ToolMessage("天气查询结果")
            → 打印: "Assistant: 天气查询结果"（实际会过滤掉非AIMessage）
        ↓
... 直到图执行到 END
```

简单说：**`stream` 是边跑边输出，`invoke` 是跑完再输出；`event` 是"某个节点跑完了"的通知；`values()` 是不关心节点名字直接拿数据；`[-1]` 是取最新生成的那条消息。**
"""

Assistant: 你好！我不知道你是谁呀～😊

在我们的对话中，我还没有了解到关于你的信息。我是小米的MiMo，很高兴认识你！

如果你愿意的话，可以告诉我一些关于你的事情吗？比如你叫什么名字，或者你今天想找我聊什么呢？
Goodbye!
